In [417]:
import pandas as pd
import numpy as np

# Import du dataset

In [418]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dhrubangtalukdar/200-years-of-global-major-earthquakes-18262026")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\trist\.cache\kagglehub\datasets\dhrubangtalukdar\200-years-of-global-major-earthquakes-18262026\versions\1


In [419]:
df = pd.read_csv(f"{path}/earthquake1826_2026.csv")

# Transformation

In [420]:
# Drop des features inutiles et magError/magNst pour éviter le leakage du model de prediction de mag
df = df.drop(["id", "status", "updated", "place", "type", "magError", "magNst"], axis=1)

In [421]:
# Comme vu dans l'exploration, les années pré 2000 possèdent trop de valeur manquante
df = df[pd.to_datetime(df['time'], format='mixed', utc=True).dt.year >= 1900]

In [422]:
df['depth'] = df['depth'].clip(lower=0)
df['rms'] = df['rms'].clip(lower=0)
df['depthError'] = df['depthError'].clip(lower=0)

In [423]:
# 10 km et 33 km sont des profondeurs par défaut dans les catalogues sismiques (depth non mesurable)
df['depth'] = df['depth'].replace({10.0: np.nan, 33.0: np.nan})

In [424]:
cols_to_cap = [c for c in df.select_dtypes(include='number').columns if c != 'mag']
for col in cols_to_cap:
    p99 = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=p99)

In [425]:
# df["time"] = pd.to_datetime(df["time"], format="mixed", utc=True)

# df['month_sin'] = np.sin(2 * np.pi * df['time'].dt.month / 12)
# df['month_cos'] = np.cos(2 * np.pi * df['time'].dt.month / 12)

# df['day_sin'] = np.sin(2 * np.pi * df['time'].dt.dayofyear / 365)
# df['day_cos'] = np.cos(2 * np.pi * df['time'].dt.dayofyear / 365)

# df['weekday_sin'] = np.sin(2 * np.pi * df['time'].dt.dayofweek / 7)
# df['weekday_cos'] = np.cos(2 * np.pi * df['time'].dt.dayofweek / 7)

In [426]:
df['lon_sin'] = np.sin(np.radians(df['longitude']))
df['lon_cos'] = np.cos(np.radians(df['longitude']))

df['lat_sin'] = np.sin(np.radians(df['latitude']))
df['lat_cos'] = np.cos(np.radians(df['latitude']))

In [427]:
# Standardisation vers Mw — formules empiriques Scordilis (2006)
mag = df['mag'].values
t = df['magType'].str.lower().str.strip()

is_mw   = t.isin(['mw', 'mww', 'mwb', 'mwc', 'mwr', 'mwp'])
is_mb   = t.isin(['mb', 'mb_lg', 'mlg', 'lg'])
is_ms_lo = t.isin(['ms', 'ms_20']) & (df['mag'] <= 6.1)
is_ms_hi = t.isin(['ms', 'ms_20']) & (df['mag'] > 6.1)
is_ml   = t.isin(['ml', 'md', 'mh', 'mc', 'ma', 'mj', 'mfa'])

df['mag'] = np.select(
    [is_mw,  is_mb,                is_ms_lo,           is_ms_hi,           is_ml],
    [mag,    1.159 * mag - 0.659,  0.67 * mag + 2.07,  0.99 * mag + 0.08,  0.67 * mag + 1.95],
    default=mag
)

In [428]:
df = df.drop(["time", 'magType', 'latitude', 'longitude'], axis=1)

## Export

In [429]:
df.head()

,depth,mag,nst,gap,dmin,rms,horizontalError,depthError,lon_sin,lon_cos,lat_sin,lat_cos
0,547.033,5.5000,63.0,39.0,5.976,1.13,8.66,7.704,-0.001773,-0.999998,-0.397680,0.917524
1,11.854,5.5000,58.0,53.0,1.485,1.40,5.21,4.721,0.798010,-0.602644,0.126827,0.991925
2,35.000,6.4000,118.0,30.0,1.178,1.17,7.94,1.842,0.801545,-0.597934,0.128626,0.991693
3,62.384,5.0000,47.0,79.0,2.388,0.54,9.64,6.266,0.203702,-0.979033,-0.302829,0.953045
4,NaN,5.3678,112.0,54.0,1.991,0.88,6.54,1.848,0.964159,0.265326,0.605326,0.795978


In [430]:
df.to_csv('../data/transformed_data.csv', index=False)